In [1]:
!pip install -q python-docx python-pptx

In [ ]:
# Шаг 1: читаем лекции и режем на куски
# В Colab сначала выполни:  !pip install -q python-docx python-pptx
import glob, os
from docx import Document
from pptx import Presentation

FOLDER = "/content"  # папка с лекциями (в Colab: "/content")
CHUNK_SIZE = 900   # символов в куске
OVERLAP = 150      # перекрытие между кусками

def read_docx(path):
    doc = Document(path)
    lines = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    # таблицы python-docx читает отдельно, добавляем их в конец
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip().replace("\n", " ") for c in row.cells]
            lines.append(" | ".join(cells))
    return "\n".join(lines)

def read_pptx(path):
    prs = Presentation(path)
    slides = []
    for i, slide in enumerate(prs.slides, 1):
        parts = []
        for shape in slide.shapes:
            if shape.has_text_frame:
                parts.append(shape.text_frame.text.strip())
        text = "\n".join(p for p in parts if p)
        if text:
            slides.append(f"[Слайд {i}] {text}")
    return "\n\n".join(slides)

def split_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        # стараемся не рвать посреди предложения
        if end < len(text):
            cut = max(text.rfind(". ", start, end), text.rfind("\n", start, end))
            if cut > start + size // 2:
                end = cut + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return [c for c in chunks if c]

all_chunks = []
for path in sorted(glob.glob(os.path.join(FOLDER, "Лекция_*"))):
    name = os.path.basename(path)
    text = read_docx(path) if path.endswith(".docx") else read_pptx(path)
    parts = split_text(text)
    for j, c in enumerate(parts):
        all_chunks.append({"id": f"{name}#{j}", "source": name, "text": c})
    print(f"{name}: {len(text)} символов -> {len(parts)} кусков")

print("\nВсего кусков:", len(all_chunks))
print("\nПример куска:\n", all_chunks[10]["text"][:400])

In [ ]:
!pip install -q google-genai chromadb

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata
import chromadb

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
EMBED_MODEL = "gemini-embedding-001"

def embed(texts, task_type):
    vectors = []
    for i in range(0, len(texts), 50):
        batch = texts[i:i+50]
        for attempt in range(5):
            try:
                res = client.models.embed_content(
                    model=EMBED_MODEL, contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type))
                vectors += [e.values for e in res.embeddings]
                break
            except Exception as e:
                print("Ошибка, повтор через 20 сек:", str(e)[:150])
                time.sleep(20)
        else:
            raise RuntimeError("Не удалось получить векторы")
        time.sleep(1)
    return vectors

texts = [c["text"] for c in all_chunks]
vectors = embed(texts, "RETRIEVAL_DOCUMENT")
print(len(vectors), "векторов, размер каждого:", len(vectors[0]))

db = chromadb.Client()
try:
    db.delete_collection("lectures")
except Exception:
    pass
col = db.create_collection("lectures", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["id"] for c in all_chunks], documents=texts,
        embeddings=vectors, metadatas=[{"source": c["source"]} for c in all_chunks])
print("В базе кусков:", col.count())

In [ ]:
def search(question, k=4):
    q = embed([question], "RETRIEVAL_QUERY")[0]
    res = col.query(query_embeddings=[q], n_results=k)
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"[{meta['source']}] расстояние {dist:.3f}\n{doc[:300]}\n---")

search("Что такое дополнительный код и зачем он нужен?")
search("Как устроен JK-триггер?")
search("Из каких полей состоит число в формате IEEE 754?")

In [9]:
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=60_000))

MODELS = ["gemini-3.6-flash", "gemini-2.5-flash", "gemini-3.5-flash"]

SYSTEM = ("Ты помощник по курсу «Организация и архитектура вычислительных систем». "
          "Отвечай только на основе приведённого контекста, на русском языке, кратко и по делу. "
          "Если в контексте нет ответа, скажи: «В лекциях этого нет». "
          "В конце укажи, из каких лекций взят ответ.")

def ask(question, k=5):
    q = embed([question], "RETRIEVAL_QUERY")[0]
    res = col.query(query_embeddings=[q], n_results=k)
    context = "\n\n".join(f"[{m['source']}]\n{d}"
                          for d, m in zip(res["documents"][0], res["metadatas"][0]))
    prompt = f"Контекст из лекций:\n{context}\n\nВопрос: {question}"
    for round_ in range(3):
        for model in MODELS:
            try:
                r = client.models.generate_content(
                    model=model, contents=prompt,
                    config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2))
                return f"{r.text}\n\n(модель: {model})"
            except Exception as e:
                print(f"  {model}: {str(e)[:80]}")
        time.sleep(10)
    return "Не удалось получить ответ: все модели недоступны"

for question in [
    "Что такое дополнительный код и зачем он нужен?",
    "Чем JK-триггер отличается от RS-триггера?",
    "Из каких полей состоит число в формате IEEE 754?",
    "Кто выиграл чемпионат мира по футболу в 2018 году?",
]:
    print("ВОПРОС:", question)
    print(ask(question))
    print("=" * 60)

ВОПРОС: Что такое дополнительный код и зачем он нужен?
**Дополнительный код (two’s complement)** — это способ представления чисел, в котором отрицательное число $x$ в $n$-разрядном поле хранится как двоичная запись значения $2^n + x$. При этом старший разряд имеет вес $-2^{n-1}$, а остальные разряды сохраняют обычные положительные веса.

**Зачем он нужен:**
Дополнительный код позволяет обслуживать и знаковую, и беззнаковую арифметику (сложение и вычитание) одним и тем же сумматором, отбрасывая перенос за границу поля. Это исключает необходимость создавать отдельную физическую схему для работы со знаковыми числами. Кроме того, он позволяет вычислять значение числа по единому правилу без отдельного разбора знака.

*Источник: Лекция_02_Представление_информации.docx*

(модель: gemini-3.6-flash)
ВОПРОС: Чем JK-триггер отличается от RS-триггера?
JK-триггер ведёт себя как RS-схема (где J — установка, K — сброс), но при одновременной подаче единиц на оба входа (J = K = 1) он не попадает в запр